In [9]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("muhammedchreiki/youtube-video-trends-and-non-trends-dataset")

print("Path to dataset files:", path)

100%|██████████| 140M/140M [00:06<00:00, 21.7MB/s] 

Extracting files...


Path to dataset files: C:\Users\Karolcia\.cache\kagglehub\datasets\muhammedchreiki\youtube-video-trends-and-non-trends-dataset\versions\1


In [10]:
import pandas as pd
import numpy as np
import pyspark as psp
import os
import ast
import seaborn as sns
import matplotlib.pyplot as plt

dataset_path = os.path.join(path,'Youtube_Videos.csv')
df = pd.read_csv(dataset_path)
df = pd.DataFrame(df)

C:\Users\Karolcia\AppData\Local\Temp\ipykernel_6808\1846439267.py:10: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(dataset_path)


In [11]:
df.describe()

,categoryId,view_count,likes,comment_count,comments_disabled,is_trending
count,390043.000000,3.900430e+05,3.900430e+05,3.900430e+05,390043.000000,390043.000000
mean,20.426179,7.046562e+06,1.534973e+05,4.490697e+03,0.018662,0.440159
std,6.524139,4.896210e+07,6.717989e+05,4.198895e+04,0.135329,0.496407
min,1.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000
25%,19.000000,2.626010e+05,5.271000e+03,1.230000e+02,0.000000,0.000000
50%,22.000000,7.643870e+05,2.356100e+04,8.530000e+02,0.000000,0.000000
75%,24.000000,2.561359e+06,7.882900e+04,2.743000e+03,0.000000,1.000000
max,30.000000,4.168919e+09,4.337832e+07,1.048303e+07,1.000000,1.000000


In [12]:
df.isna().sum()

video_id                  0
title                     0
publishedAt               0
channelId                 0
channelTitle              0
categoryId                0
trending_date        218362
tags                 120820
view_count                0
likes                     0
comment_count             0
thumbnail_link            0
comments_disabled         0
description           59300
is_trending               0
dtype: int64

In [ ]:
df_1 = df.copy()
df_1['tags'] = df_1['tags'].notna().astype(int)
df_1 = df_1.drop(columns=['description','thumbnail_link', 'title', 'channelId'])
df_1['publishedAt'] = pd.to_datetime(df_1['publishedAt']).dt.date

df_1.head()

,video_id,publishedAt,channelTitle,categoryId,trending_date,tags,view_count,likes,comment_count,comments_disabled,is_trending
0,G4M_621v1As,2025-04-12,Vk_07_rider,22.0,2025-11-08,1,125784084.0,1557178.0,1583.0,0,1
1,z2voqo_Jhx4,2025-04-06,Leonardo Patrick,10.0,2025-11-08,0,94744011.0,925529.0,5849.0,0,1
2,jHIt9oHFLsw,2025-04-06,Violin Phonix,22.0,2025-11-08,0,61945818.0,1067412.0,4124.0,0,1
3,gwRqLbWqKlM,2025-03-19,LLOUD Official,10.0,2025-11-08,1,14555963.0,455816.0,20396.0,0,1
4,prpRoyrutcE,2025-04-14,Mr.KiranJ,10.0,2025-11-08,0,26204942.0,336267.0,2333.0,0,1


In [25]:
df_1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 390043 entries, 0 to 390042
Data columns (total 11 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   video_id           390043 non-null  object 
 1   publishedAt        390043 non-null  object 
 2   channelTitle       390043 non-null  object 
 3   categoryId         390043 non-null  float64
 4   trending_date      171681 non-null  object 
 5   tags               390043 non-null  int64  
 6   view_count         390043 non-null  float64
 7   likes              390043 non-null  float64
 8   comment_count      390043 non-null  float64
 9   comments_disabled  390043 non-null  int64  
 10  is_trending        390043 non-null  int64  
dtypes: float64(4), int64(3), object(4)
memory usage: 32.7+ MB


In [26]:
df_1.isna().sum()

video_id                  0
publishedAt               0
channelTitle              0
categoryId                0
trending_date        218362
tags                      0
view_count                0
likes                     0
comment_count             0
comments_disabled         0
is_trending               0
dtype: int64

In [51]:
numeric_cols = df_1.select_dtypes(include=[np.number]).columns
df_1.groupby(['categoryId'])[numeric_cols].agg(['sum'])

,categoryId,tags,view_count,likes,comment_count,comments_disabled,is_trending
,sum,sum,sum,sum,sum,sum,sum
categoryId,,,,,,,
1.0,14333.0,10865,1.067402e+11,1.785334e+09,56291490.0,453,7052
2.0,12210.0,4691,9.015299e+09,2.659237e+08,12974735.0,15,3675
10.0,381570.0,31680,4.753845e+11,9.113718e+09,618695824.0,182,25702
15.0,32085.0,1367,2.232507e+10,3.721182e+08,3871091.0,21,724
17.0,519231.0,24904,1.226980e+11,2.634845e+09,80302940.0,101,20622
19.0,130036.0,4341,3.221847e+10,1.014336e+09,11479348.0,45,1330
20.0,963380.0,39238,1.116704e+11,4.113619e+09,223052082.0,468,35568
22.0,1483680.0,27685,4.930636e+11,9.989025e+09,136818711.0,2023,15269


In [59]:
df[df['categoryId'] == 30].head(20)

,video_id,title,publishedAt,channelId,channelTitle,categoryId,trending_date,tags,view_count,likes,comment_count,thumbnail_link,comments_disabled,description,is_trending
172505,W5CT4ahnrnA,They Live,2024-08-01T04:00:04Z,UCuVPpxrm2VAgpH3Ktln4HXg,YouTube Movies,30.0,NaN,NaN,0.0,36674.0,5621.0,https://i.ytimg.com/vi/W5CT4ahnrnA/default.jpg,0,"Master of fright, John Carpenter directs this ...",0
172760,zQCJQuingr8,Wicked,2024-12-26T05:02:02Z,UC_5HAqApIJqWJ3735hkegzA,YouTube Movies,30.0,NaN,NaN,0.0,17338.0,1768.0,https://i.ytimg.com/vi/zQCJQuingr8/default.jpg,0,"Wicked, one of the most beloved and enduring m...",0
173331,1F7zdBmlqf8,Chain Reactions,2025-10-21T04:02:01Z,UCQYRqOrigxt0CAY10JyQing,YouTube Movies,30.0,NaN,NaN,0.0,3.0,0.0,https://i.ytimg.com/vi/1F7zdBmlqf8/default.jpg,0,Fifty years after Tobe Hooper's The Texas Chai...,0
173817,6w3iElemSYI,Limitless,2024-12-01T05:00:21Z,UCuVPpxrm2VAgpH3Ktln4HXg,YouTube Movies,30.0,NaN,NaN,0.0,39792.0,3276.0,https://i.ytimg.com/vi/6w3iElemSYI/default.jpg,0,A mysterious pill that enables the user to acc...,0
173845,A1hFIVOEcUU,Mystery Science Theater 3000: The Shape of Thi...,2023-09-22T04:00:04Z,UCuVPpxrm2VAgpH3Ktln4HXg,YouTube Movies,30.0,NaN,NaN,0.0,347.0,77.0,https://i.ytimg.com/vi/A1hFIVOEcUU/default.jpg,0,Emily's crew watch three humans and their robo...,0
173870,IKI9J_0NP_c,Brother Nature,2025-07-01T04:00:20Z,UCuVPpxrm2VAgpH3Ktln4HXg,YouTube Movies,30.0,NaN,NaN,0.0,166.0,19.0,https://i.ytimg.com/vi/IKI9J_0NP_c/default.jpg,0,Roger (Taran Killam) a straight-laced politici...,0
174977,86M_Td6fw1I,"Top 30 Alien Encounters, Technologies and Abdu...",2024-08-23T04:00:21Z,UCuVPpxrm2VAgpH3Ktln4HXg,YouTube Movies,30.0,NaN,NaN,0.0,923.0,85.0,https://i.ytimg.com/vi/86M_Td6fw1I/default.jpg,0,Harrowing accounts showcasing a plethora of al...,0
174982,Kc3JW26GDwU,Tech Billionaires: Bill Gates,2021-05-04T04:02:03Z,UCz3O7eG2klRtQIpczVsqBLQ,YouTube Movies,30.0,NaN,NaN,0.0,1.0,4.0,https://i.ytimg.com/vi/Kc3JW26GDwU/default.jpg,0,"Bill Gates, the extraordinarily successful Ame...",0
175305,Q0SoGkNC-r4,Fashion Reimagined,2024-06-17T04:00:01Z,UCuVPpxrm2VAgpH3Ktln4HXg,YouTube Movies,30.0,NaN,NaN,0.0,1700.0,86.0,https://i.ytimg.com/vi/Q0SoGkNC-r4/default.jpg,0,After winning the coveted Vogue award for the ...,0
175354,7J4BMLTcCCw,Exam,2023-10-06T04:00:03Z,UCuVPpxrm2VAgpH3Ktln4HXg,YouTube Movies,30.0,NaN,NaN,0.0,1165.0,173.0,https://i.ytimg.com/vi/7J4BMLTcCCw/default.jpg,0,Eight candidates have reached the final stage ...,0


In [ ]:
def map_category(category_id):
    if category_id == 1:
        return 'mix'
    elif category_id == 2:
        return 'insta_style'
    elif category_id == 10:
        return 'miusic'
    elif category_id == 15:
        return 'funny'
    elif category_id == 17:
        return 'sport'
    elif category_id == 19:
        return 'tops_lists'
    elif category_id == 20:
        return 'games'
    elif category_id == 22:
        return 'mix2'
    elif category_id == 23:
        return 'funny_mix'
    elif category_id == 24:
        return 'film'
    elif category_id == 25:
        return 'news'
    elif category_id == 26:
        return 'generar_know'
    elif category_id == 27:
        return 'kid_know'
    elif category_id == 28:
        return 'generar_know'
    elif category_id == 29:
        return 'generar_know'
    elif category_id == 30:
        return 'not_shared'
    else:
        return f'category_{category_id}'

df_1['categoryName'] = df_1['categoryId'].apply(map_category)

In [ ]:
numeric_cols = df_1.select_dtypes(include=[np.number]).columns
df_1.groupby(['categoryId'])[numeric_cols].agg(['sum', 'mean'])

SyntaxError: invalid syntax (3280822675.py, line 1)